# Local volatility

`LocalVolatilityModel` on TICKER1: from the implied surface to the local vol (Gatheral's formula in total variance),
static-arbitrage checks, the classic sanity checks, then Monte Carlo (forward, smile repriced, greeks by AAD).
Docs: *Models and Pricing* (Models: Local volatility), *Architecture* (Models).

In [ ]:
import os
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorquant as tq
from scipy.optimize import brentq
from scipy.stats import norm

## Market
Implied vols on maturities $\times$ absolute strikes; the model works in forward log-moneyness $y = \ln(K/F(T))$.

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date
TICKER = "TICKER1"
eq = mkt.get_equity(TICKER)
vs = eq.vol_surface
S0 = float(eq.spot)
T_NODES = np.array(vs.maturity, dtype=float)
K_NODES = np.array(vs.strike, dtype=float)
IV = vs.volatility_matrix.numpy()


def date_of(T):
    return EVALUATION_DATE + timedelta(days=max(1, int(round(T * 365))))


F_NODES = np.array([float(eq.forward(date_of(T))) for T in T_NODES])
Y_NODES = np.log(K_NODES[None, :] / F_NODES[:, None])            # [nT, nK] forward log-moneyness
print(f"{TICKER} spot {S0:.2f}, dividends: {eq.dividend_model}")
print(pd.DataFrame(100 * IV, index=np.round(T_NODES, 2), columns=[f"{k:.0%}" for k in K_NODES / S0]).round(1).to_string())

## Total variance
$w(y, T) = \sigma_{\mathrm{imp}}^2 T$. Per maturity a natural cubic spline in $y$ through the nodes, linear outside (wing slope
clipped to 1.9 < 2, Lee); in $T$ linear at fixed $y$. No calendar arbitrage: the slices must not cross ($\partial_T w > 0$).

In [ ]:
y = np.linspace(-0.8, 0.6, 281)
W = np.stack([tq.natural_cubic_spline(Y_NODES[i], IV[i] ** 2 * T_NODES[i], y, 1.9)[0].numpy()
              for i in range(len(T_NODES))])
fig, ax = plt.subplots(figsize=(8, 4))
for i, T in enumerate(T_NODES):
    ax.plot(y, W[i], label=f"T = {T:.2f}")
    ax.plot(Y_NODES[i], IV[i] ** 2 * T, "k.", ms=4)
ax.set(xlabel="y = ln(K / F)", ylabel="w = iv² T", title="Total implied variance (dots: nodes)")
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.3)
plt.show()
print("calendar check, min over y of w(T_i+1) - w(T_i):", np.round((W[1:] - W[:-1]).min(axis=1), 5))

## Local vol (Gatheral)
$$\sigma_{\mathrm{loc}}^2 = \frac{\partial_T w}{g},\qquad
g = 1 - \frac{y}{w}\,\partial_y w + \frac14\Big(-\frac14 - \frac1w + \frac{y^2}{w^2}\Big)(\partial_y w)^2 + \frac12\,\partial_{yy} w .$$
$g > 0$ is the absence of butterfly arbitrage (a positive density), $\partial_T w > 0$ of calendar arbitrage. The model
reports both on the data region.

In [ ]:
model = tq.LocalVolatilityModel.from_equity(eq)
print(model.arbitrage)
sigma, t_lv, y_lv = (v.numpy() for v in model.local_vol_grid)
print(f"grid: {len(t_lv)} times (step 1/96 Y), {len(y_lv)} points of y in [{y_lv[0]:.2f}, {y_lv[-1]:.2f}]")

In [ ]:
sel = (y_lv > -0.8) & (y_lv < 0.6)
fig, ax = plt.subplots(figsize=(8, 4))
mesh = ax.pcolormesh(y_lv[sel], t_lv, sigma[:, sel], shading="auto", cmap="viridis", vmax=1.0)
ax.plot(Y_NODES.ravel(), np.repeat(T_NODES, len(K_NODES)), "w.", ms=3)
fig.colorbar(mesh, label="local vol")
ax.set(xlabel="y = ln(S / F(t))", ylabel="t", title="Local vol surface (dots: implied vol nodes)")
plt.show()

## Sanity checks
1. **Skew**: the local vol smile is steeper than the implied one. The rule of thumb is a local skew **twice** the implied
   one at the money ($\partial_y \sigma_{\mathrm{imp}} \approx \frac12 \partial_y \sigma_{\mathrm{loc}}$: the implied vol averages the local vol between the spot
   and the strike); it holds for a linear skew at short maturity, the convexity of this smile lowers the ratio.
2. **ATM level**: $\sigma_{\mathrm{loc}}(t, 0) \to \sigma_{\mathrm{imp}}(\text{ATM})$ only as $T \to 0$; with a convex smile the local vol at the money
   is below the implied one (the curvature term $\frac12 \partial_{yy} w$ in $g$: the ATM option also sees the higher local vols of
   the wings).
3. **Flat smile**: a flat implied surface gives a flat local vol equal to it.

In [ ]:
def lv_row(T):
    return sigma[np.argmin(np.abs(t_lv - T))]


fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, i in zip(axes, (1, 3, 6)):
    T = T_NODES[i]
    iv_slice = np.sqrt(np.maximum(W[i], 1e-12) / T)
    ax.plot(y, iv_slice, label="implied")
    ax.plot(y_lv[sel], lv_row(T)[sel], label="local")
    ax.plot(Y_NODES[i], IV[i], "k.", ms=5)
    ax.set(title=f"T = {T:.2f}", xlabel="y")
    ax.grid(alpha=0.3)
axes[0].legend()
plt.show()

rows = []
for i, T in enumerate(T_NODES[:5]):
    iv_s = np.sqrt(np.maximum(W[i], 1e-12) / T)
    j = np.argmin(np.abs(y)); k = np.argmin(np.abs(y_lv))
    skew_iv = (iv_s[j + 5] - iv_s[j - 5]) / (y[j + 5] - y[j - 5])
    row = lv_row(T)
    skew_lv = (row[k + 2] - row[k - 2]) / (y_lv[k + 2] - y_lv[k - 2])
    rows.append((T, iv_s[j], row[k], skew_iv, skew_lv, skew_lv / skew_iv))
print(pd.DataFrame(rows, columns=["T", "ATM implied", "ATM local", "implied skew", "local skew", "ratio"]).round(3).to_string(index=False))

In [ ]:
# flat smile: the same Gatheral routine on a constant 25% surface
flat = np.full_like(IV, 0.25)
sigma_flat, _, diag_flat = tq.gatheral_local_vol(flat, T_NODES, tf.constant(Y_NODES), y_lv)
print(f"flat 25% surface -> local vol in [{sigma_flat.numpy().min():.6f}, {sigma_flat.numpy().max():.6f}]", diag_flat)

## Implied vol as an average of local vol
Gatheral (3.8)-(3.11): the implied variance is the time average of the local variance over the paths that end at the
strike, weighted by a Brownian-bridge density from the spot to the strike (centre $\tfrac tT\,y_K$ in forward log-moneyness,
variance $\sigma^2 t (T-t)/T$):
$$\sigma_{\mathrm{imp}}^2(K, T) \approx \frac1T\int_0^T \E_{\text{bridge}}\big[\sigma_{\mathrm{loc}}^2(t, y_t)\big]\,\mathrm{d}t
\;\approx\; \frac1T\int_0^T \sigma_{\mathrm{loc}}^2\big(t, \tfrac tT\,y_K\big)\,\mathrm{d}t\ \ \text{(peak only)}.$$
The peak-only version explains the skew rule (the path to a strike only gets half-way in $y$ on average) and why
short-dated wings need far higher local vols; it misses the convexity of the local variance around the path, which the
bridge average restores: within about 1 vol point near the money (0.3-1), cruder in the short-dated downside wing,
where a Gaussian bridge in $y$ is a poor picture of the paths.

In [ ]:
GH_X, GH_W = np.polynomial.hermite_e.hermegauss(40)                # Gauss-Hermite for a standard normal
GH_W = GH_W / GH_W.sum()


def path_average(T, yK, iv, bridge=True, n=200):
    total = 0.0
    for t in (np.arange(n) + 0.5) / n * T:
        row = sigma[np.searchsorted(t_lv, t)]
        sd = iv * np.sqrt(t * (T - t) / T) if bridge else 0.0
        total += np.sum(GH_W * np.interp(t / T * yK + sd * GH_X, y_lv, row) ** 2)
    return np.sqrt(total / n)


for bridge in (False, True):
    rows = {f"T={T_NODES[i]:.2f}": [100 * (path_average(T_NODES[i], yk, iv, bridge) - iv) for yk, iv in zip(Y_NODES[i], IV[i])]
            for i in (1, 3, 5, 6)}
    print(("bridge average" if bridge else "peak only") + " - market implied vol (vol points), by strike / spot:")
    print(pd.DataFrame(rows, index=[f"{k:.0%}" for k in K_NODES / S0]).T.round(2).to_string(), "\n")

## Monte Carlo
$X_t = S_t / F(t)$ with $\mathrm{d}X/X = \sigma_{\mathrm{loc}}(t, \ln X)\,\mathrm{d}W$ (log-Euler), $S_t = F(t)\,X_t$: the forward is the market one
at every date, dividends included. 20k paths, 50 steps a year, observed at the surface maturities.

In [ ]:
N_PATHS, STEPS = 20_000, 50
T_MAX = T_NODES[-2]                                       # up to 3Y
obs = [T for T in T_NODES if T <= T_MAX]
grid = np.unique(np.concatenate([np.arange(1, int(STEPS * T_MAX) + 1) / STEPS, obs]))
dw = tf.random.stateless_normal([N_PATHS, len(grid)], seed=[42, 0], dtype=tf.float64)
paths = model.evolve(grid, dw).numpy()                    # [paths, steps], martingale correction on
raw = tq.LocalVolatilityModel.from_equity(eq, martingale_correction=False).evolve(grid, dw).numpy()

fwd = np.array([float(eq.forward(date_of(t))) for t in grid])
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid, paths[:100].T, color="steelblue", alpha=0.15, lw=0.6)
ax.plot(grid, raw.mean(axis=0), color="tomato", lw=2, label="MC mean (no correction)")
ax.plot(grid, fwd, "k--", lw=1.5, label="Equity.forward")
ax.set(xlabel="t", ylabel="S", title="Local vol paths and forward (drops at the ex-dates)",
       ylim=(0.3 * S0, 2.2 * S0))
ax.legend()
plt.show()
cols = [int(np.argmin(np.abs(grid - T))) for T in obs]
err = (raw[:, cols].mean(axis=0) - fwd[cols]) / (raw[:, cols].std(axis=0) / np.sqrt(N_PATHS))
print("forward error without correction, in standard errors:", np.round(err, 2))
print("with correction (exact):", np.abs(paths[:, cols].mean(axis=0) / fwd[cols] - 1).max())

## The smile repriced
OTM options on the simulated paths (puts below the forward, calls above) back to implied vols, against the market nodes:
the local vol model reprices the surface it was built from, up to the Monte Carlo error (~0.1-0.2 vol points
here) and a small positive bias of the discretisation (log-Euler, 50 steps a year, local vol bilinear on its grid).

In [ ]:
def black_otm(F, K, s, T, DF):
    d1 = (np.log(F / K) + 0.5 * s * s * T) / (s * np.sqrt(T))
    call = DF * (F * norm.cdf(d1) - K * norm.cdf(d1 - s * np.sqrt(T)))
    return call - DF * (F - K) if K < F else call


def implied(price, F, K, T, DF):
    try:
        return brentq(lambda s: black_otm(F, K, s, T, DF) - price, 1e-4, 3.0)
    except ValueError:
        return np.nan


rows = {}
for i, T in enumerate(obs):
    if T < 0.4:
        continue
    s_T, F, DF = paths[:, cols[i]], F_NODES[i], float(eq.discount_curve.discount(date_of(T)))
    mc = [implied(DF * np.mean(np.maximum(K - s_T, 0) if K < F else np.maximum(s_T - K, 0)), F, K, T, DF) for K in K_NODES]
    rows[f"T={T:.2f}"] = 100 * (np.array(mc) - IV[i])
print("MC implied - market (vol points), by strike / spot:")
print(pd.DataFrame(rows, index=[f"{k:.0%}" for k in K_NODES / S0]).T.round(2).to_string())

## Forward smile
A forward-start call pays $(S_{T_2}/S_{T_1} - k)^+$: its implied vol (on the forward ratio $F(T_2)/F(T_1)$) is the smile the
model predicts for one year starting in one year. Local vol flattens it: the local skew decays with time, so the
forward smile from 1Y to 2Y is much flatter than today's 1Y smile, while stochastic vol keeps it (Heston, SLV).

In [ ]:
i1, i2 = list(T_NODES).index(1.0), int(np.argmin(np.abs(T_NODES - 2.0)))
ratio = paths[:, cols[i2]] / paths[:, cols[i1]]
f_ratio = F_NODES[i2] / F_NODES[i1]
dT = T_NODES[i2] - T_NODES[i1]
ks = np.array([0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3])
fwd_smile = [implied(np.mean(np.maximum(k * f_ratio - ratio, 0) if k < 1 else np.maximum(ratio - k * f_ratio, 0)),
                     f_ratio, k * f_ratio, dT, 1.0) for k in ks]
spot_smile = [float(np.sqrt(np.interp(np.log(k), y, W[i1]) / T_NODES[i1])) for k in ks]
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(ks, 100 * np.array(spot_smile), "o-", label="today's 1Y smile (market)")
ax.plot(ks, 100 * np.array(fwd_smile), "s-", label="1Y forward smile in 1Y (local vol MC)")
ax.set(xlabel="moneyness k (strike / forward)", ylabel="implied vol (%)", title="Forward smile flattening")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Greeks by AAD
`VanillaMCPricer(model=LocalVolatilityModel)`: the model is rebuilt from the market inside the tape, so the pathwise
delta and the vega **per vol node** come out of one backward pass. The local vol reprices every vanilla, so with the
surface held in absolute strikes its delta is the Black–Scholes sticky-strike one with the **same dividend model** (cash
dividends: `dividend_model="discrete"`; the default continuous model holds the implied yield and gives a smaller delta).
Against Black–Scholes:
the total vega is close, but the local vol puts it on the 1Y nodes around the strike with small +/- contributions on
the neighbours and on the earlier slices (a node moves the local vol, through $\partial_T w$ and the spline, on a region).

In [ ]:
expiry = date_of(1.0)
K = round(float(eq.forward(expiry)), 2)


def call():
    return tq.VanillaOption(eq.currency, start_date=EVALUATION_DATE, end_date=expiry, option_type=tq.OptionType.Call,
                            strike=K, underlying=TICKER)


lv_call, bs_call = call(), call()
lv = tq.VanillaMCPricer(model=tq.LocalVolatilityModel, n_paths=20_000, n_steps=50, seed=7)
lv.price(lv_call, mkt, autodiff=True)
bs = tq.BlackScholesPricer(dividend_model="discrete")
bs.price(bs_call, mkt, autodiff=True)
sources = [eq.spot, vs.volatility_matrix]
(d_lv, v_lv), (d_bs, v_bs) = lv.tape.gradient(lv_call.price, sources), bs.tape.gradient(bs_call.price, sources)
print(f"price  LV {float(lv_call.price):.3f} +/- {float(lv_call.standard_error):.3f}   BS {float(bs_call.price):.3f}")
print(f"delta  LV {float(d_lv):.4f}   BS {float(d_bs):.4f}")
print(f"vega (sum over nodes, per vol point)  LV {float(tf.reduce_sum(v_lv)) / 100:.3f}   BS {float(tf.reduce_sum(v_bs)) / 100:.3f}")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), sharey=True)
for ax, v, name in ((axes[0], v_lv, "local vol (MC, AAD)"), (axes[1], v_bs, "Black-Scholes")):
    m = ax.imshow(v.numpy() / 100, aspect="auto", cmap="RdBu_r", vmin=-0.4, vmax=0.4)
    ax.set(title=f"vega per node: {name}", xticks=range(len(K_NODES)), xticklabels=[f"{k:.0%}" for k in K_NODES / S0],
           yticks=range(len(T_NODES)), yticklabels=np.round(T_NODES, 2), xlabel="K / S0")
axes[0].set_ylabel("T")
fig.colorbar(m, ax=axes)
plt.show()

## Exercises
1. Bump one node of the 6M slice by 1 vol point: where does the local vol move (plot the difference of the grids)?
2. Replace 50 by 12 steps a year in the Monte Carlo: how much worse is the 1Y smile in the wings?
3. Why is the local vol at a short maturity in the far wings so much higher than the implied vol there (use the path
   average above)?